# Algoritmo Genético para el TSP
## 03MIAR - Algoritmos de Optimización · Actividad Guiada 3

**Nombre:** Donald Silva

**Link:** `https://colab.research.google.com/drive/1O4qwDgZwdJBiM-osr69NsfD5onWZTTlZ?usp=sharing`

**Github:** `https://github.com/dsilvas2001/03MIAR_DonaldSilva_AG_.git`

---

Este notebook completa el esqueleto de algoritmo genético que acompaña a la AG3: reutiliza sus
funciones (`crear_solucion`, `distancia_total`) y deja **siete funciones sin cuerpo** que hay que
implementar. La función principal `algoritmo_genetico()` ya viene escrita y es la que define el
contrato que deben cumplir.

### Las siete funciones a implementar

| Función | Cometido |
|---|---|
| `generar_poblacion(Nodos, N)` | Población inicial de N soluciones |
| `Evaluar_Poblacion(poblacion, problem)` | Devolver el mejor individuo y su distancia |
| `Cruzar(poblacion, mutacion, problem)` | Cruzar la población y devolverla ampliada con los hijos |
| `Descendencia(padres, problem, mutacion)` | Generar hijos por cruce de un punto |
| `Factibilizar(solucion, problem)` | Reparar hijos: el cruce repite ciudades y omite otras |
| `Mutar(solucion, mutacion)` | Alterar un individuo con probabilidad `mutacion` |
| `Seleccionar(problem, poblacion, N, elitismo)` | Reducir a N individuos respetando el elitismo |

### Las cuatro mejoras que sugieren los comentarios

1. *«Podría aplicarse un proceso previo de selección para elegir los individuos que se desea
   cruzar»* → emparejamiento por **torneo**.
2. *«Es posible usar otros n-puntos, uniforme»* → cruce de **uno y de dos puntos**.
3. *«Se podrían añadir otros operadores»* de mutación → **intercambio** e **inversión**.
4. *Usar una selección de ruleta (proporcional a su fitness)* para los no élite.

Al final se mide cuánto aporta cada una.

---
## Carga de datos del problema

Misma cascada que en el notebook de la AG3, por el aviso del enunciado sobre la descarga: se prueba
el fichero **`swiss42.tsp` local**, luego la **descarga desde TSPLIB** y, si nada funciona, una
**copia de la matriz incrustada** aquí mismo. Así el notebook arranca siempre.

In [1]:
import math
import random
import time

random.seed(42)


class ProblemaMatriz:
    """Instancia del TSP a partir de una matriz de distancias.

    Misma interfaz que un objeto de tsplib95: get_nodes() y get_weight(a, b).
    """

    def __init__(self, matriz, nombre=""):
        self.matriz = matriz
        self.nombre = nombre

    def get_nodes(self):
        return list(range(len(self.matriz)))

    def get_weight(self, a, b):
        return self.matriz[a][b]


def parsear_tsp(texto):
    """Convierte el contenido de un fichero TSPLIB en una matriz de distancias."""
    cabecera = {}
    for linea in texto.splitlines():
        if ":" in linea and "SECTION" not in linea:
            clave, _, valor = linea.partition(":")
            cabecera[clave.strip().upper()] = valor.strip()

    if "EDGE_WEIGHT_SECTION" in texto:
        cuerpo = texto.split("EDGE_WEIGHT_SECTION")[1].replace("EOF", "")
        valores = [int(float(x)) for x in cuerpo.split()]
        n = int(cabecera["DIMENSION"])
        return [valores[i * n:(i + 1) * n] for i in range(n)]

    tipo = cabecera.get("EDGE_WEIGHT_TYPE", "EUC_2D").upper()
    cuerpo = texto.split("NODE_COORD_SECTION")[1].replace("EOF", "")

    puntos = []
    for linea in cuerpo.strip().splitlines():
        partes = linea.split()
        if len(partes) >= 3:
            puntos.append((float(partes[1]), float(partes[2])))

    n = len(puntos)
    matriz = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            dx = puntos[i][0] - puntos[j][0]
            dy = puntos[i][1] - puntos[j][1]
            if tipo == "ATT":
                r = math.sqrt((dx * dx + dy * dy) / 10.0)
                t = int(round(r))
                matriz[i][j] = t + 1 if t < r else t
            else:
                matriz[i][j] = int(round(math.sqrt(dx * dx + dy * dy)))

    return matriz


def cargar_problema(instancia="swiss42", avisar=True):
    """Carga la instancia probando: fichero local -> descarga -> copia incrustada."""
    fichero = instancia + ".tsp"

    try:
        with open(fichero, encoding="utf-8") as f:
            texto = f.read()
        if avisar:
            print(f"Datos leídos del fichero local '{fichero}'")
        return ProblemaMatriz(parsear_tsp(texto), instancia)
    except OSError:
        pass

    try:
        import gzip
        import urllib.request
        url = f"http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/{instancia}.tsp.gz"
        with urllib.request.urlopen(url, timeout=10) as respuesta:
            texto = gzip.decompress(respuesta.read()).decode("utf-8")
        with open(fichero, "w", encoding="utf-8") as f:
            f.write(texto)
        if avisar:
            print(f"Datos descargados de TSPLIB y guardados en '{fichero}'")
        return ProblemaMatriz(parsear_tsp(texto), instancia)
    except Exception as error:
        if avisar:
            print(f"La descarga ha fallado ({type(error).__name__}).")

    if instancia != "swiss42":
        raise RuntimeError(f"No hay copia incrustada de '{instancia}'.")
    if avisar:
        print("Se usa la copia de la matriz incrustada en el notebook")
    matriz = [[int(v) for v in fila.split()] for fila in MATRIZ_SWISS42.strip().splitlines()]
    return ProblemaMatriz(matriz, instancia)


MATRIZ_SWISS42 = """
    0 15 30 23 32 55 33 37 92 114 92 110 96 90 74 76 82 67 72 78 82 159 122 131 206 112 57 28 43 70 65 66 37 103 84 125 129 72 126 141 183 124
    15 0 34 23 27 40 19 32 93 117 88 100 87 75 63 67 71 69 62 63 96 164 132 131 212 106 44 33 51 77 75 72 52 118 99 132 132 67 139 148 186 122
    30 34 0 11 18 57 36 65 62 84 64 89 76 93 95 100 104 98 57 88 99 130 100 101 179 86 51 4 18 43 45 95 45 115 93 152 159 100 112 114 153 94
    23 23 11 0 11 48 26 54 70 94 69 89 75 84 84 89 92 89 54 78 99 141 111 109 190 89 44 11 29 54 56 89 47 118 96 147 151 90 122 126 163 101
    32 27 18 11 0 40 20 58 67 92 61 78 65 76 83 89 91 95 43 72 110 141 116 105 190 81 34 19 35 57 63 97 58 129 107 156 158 92 129 127 161 95
    55 40 57 48 40 0 23 55 96 123 78 75 62 36 56 66 63 95 37 34 137 174 156 129 224 90 15 59 75 96 103 105 91 158 139 164 156 78 169 163 191 115
    33 19 36 26 20 23 0 45 85 111 75 82 69 60 63 70 71 85 44 52 115 161 136 122 210 91 25 37 54 78 81 90 68 136 116 150 147 76 148 147 180 111
    37 32 65 54 58 55 45 0 124 149 118 126 113 80 42 42 49 40 87 60 94 195 158 163 242 135 65 63 79 106 101 50 66 118 104 109 103 36 160 178 218 153
    92 93 62 70 67 96 85 124 0 28 29 68 63 122 148 155 156 159 67 129 148 78 80 39 129 46 82 65 55 40 61 157 97 159 135 212 221 159 110 72 95 35
    114 117 84 94 92 123 111 149 28 0 54 91 88 150 174 181 182 181 95 157 159 50 65 27 102 65 110 87 73 50 68 176 112 166 142 229 241 184 99 46 69 38
    92 88 64 69 61 78 75 118 29 54 0 39 34 99 134 142 141 157 44 110 161 103 109 52 154 22 63 68 66 61 81 158 107 175 151 216 219 150 137 100 115 37
    110 100 89 89 78 75 82 126 68 91 39 0 14 80 129 139 135 167 39 98 187 136 148 81 186 28 61 92 97 98 117 173 134 204 181 232 229 153 176 137 143 62
    96 87 76 75 65 62 69 113 63 88 34 14 0 72 117 128 124 153 26 88 174 136 142 82 187 32 48 79 85 89 106 159 121 191 168 219 216 140 168 134 145 64
    90 75 93 84 76 36 60 80 122 150 99 80 72 0 59 71 63 116 56 25 170 201 189 151 252 104 44 95 111 130 138 130 127 192 174 186 172 90 205 193 214 135
    74 63 95 84 83 56 63 42 148 174 134 129 117 59 0 11 8 63 93 35 135 223 195 184 273 146 71 95 113 138 138 81 107 159 146 132 113 32 200 209 243 171
    76 67 100 89 89 66 70 42 155 181 142 139 128 71 11 0 11 54 103 46 130 230 198 192 279 155 80 99 117 143 141 74 107 155 143 122 102 22 202 215 250 179
    82 71 104 92 91 63 71 49 156 182 141 135 124 63 8 11 0 65 100 39 140 232 203 192 281 153 78 103 121 147 146 85 115 164 152 133 112 33 208 218 251 178
    67 69 98 89 95 95 85 40 159 181 157 167 153 116 63 54 65 0 127 92 83 224 180 199 269 175 106 95 109 135 125 21 80 107 100 71 63 33 173 205 249 191
    72 62 57 54 43 37 44 87 67 95 44 39 26 56 93 103 100 127 0 67 153 145 139 96 196 53 23 60 70 81 95 134 101 172 149 194 190 115 160 138 159 80
    78 63 88 78 72 34 52 60 129 157 110 98 88 25 35 46 39 92 67 0 152 207 188 162 258 119 48 89 107 129 134 108 114 176 159 163 147 66 200 197 224 147
    82 96 99 99 110 137 115 94 148 159 161 187 174 170 135 130 140 83 153 152 0 188 128 184 222 183 139 95 95 110 91 62 54 24 23 81 110 113 108 164 217 184
    159 164 130 141 141 174 161 195 78 50 103 136 136 201 223 230 232 224 145 207 188 0 65 57 51 109 160 132 116 90 102 217 148 188 168 264 281 231 100 26 30 75
    122 132 100 111 116 156 136 158 80 65 109 148 142 189 195 198 203 180 139 188 128 65 0 91 94 126 145 100 82 60 57 167 99 126 106 208 230 194 36 39 94 103
    131 131 101 109 105 129 122 163 39 27 52 81 82 151 184 192 192 199 96 162 184 57 91 0 106 53 115 104 94 74 94 196 134 192 168 251 260 197 126 64 64 19
    206 212 179 190 190 224 210 242 129 102 154 186 187 252 273 279 281 269 196 258 222 51 94 106 0 158 211 180 163 136 145 259 190 218 200 302 323 278 120 65 49 124
    112 106 86 89 81 90 91 135 46 65 22 28 32 104 146 155 153 175 53 119 183 109 126 53 158 0 75 89 88 83 103 178 129 197 173 236 238 166 156 111 115 34
    57 44 51 44 34 15 25 65 82 110 63 61 48 44 71 80 78 106 23 48 139 160 145 115 211 75 0 53 68 86 95 114 90 160 139 173 168 92 162 150 176 101
    28 33 4 11 19 59 37 63 65 87 68 92 79 95 95 99 103 95 60 89 95 132 100 104 180 89 53 0 18 44 45 92 42 112 89 149 156 99 111 116 155 97
    43 51 18 29 35 75 54 79 55 73 66 97 85 111 113 117 121 109 70 107 95 116 82 94 163 88 68 18 0 27 27 103 42 109 85 157 168 115 94 98 140 90
    70 77 43 54 57 96 78 106 40 50 61 98 89 130 138 143 147 135 81 129 110 90 60 74 136 83 86 44 27 0 21 128 62 119 96 179 192 142 79 72 115 74
    65 75 45 56 63 103 81 101 61 68 81 117 106 138 138 141 146 125 95 134 91 102 57 94 145 103 95 45 27 21 0 115 46 98 75 163 179 136 67 81 129 95
    66 72 95 89 97 105 90 50 157 176 158 173 159 130 81 74 85 21 134 108 62 217 167 196 259 178 114 92 103 128 115 0 69 86 81 60 65 54 158 195 243 190
    37 52 45 47 58 91 68 66 97 112 107 134 121 127 107 107 115 80 101 114 54 148 99 134 190 129 90 42 42 62 46 69 0 71 49 117 133 98 95 127 175 132
    103 118 115 118 129 158 136 118 159 166 175 204 191 192 159 155 164 107 172 176 24 188 126 192 218 197 160 112 109 119 98 86 71 0 24 94 127 137 100 163 218 194
    84 99 93 96 107 139 116 104 135 142 151 181 168 174 146 143 152 100 149 159 23 168 106 168 200 173 139 89 85 96 75 81 49 24 0 104 133 127 85 143 197 170
    125 132 152 147 156 164 150 109 212 229 216 232 219 186 132 122 133 71 194 163 81 264 208 251 302 236 173 149 157 179 163 60 117 94 104 0 39 100 190 241 292 246
    129 132 159 151 158 156 147 103 221 241 219 229 216 172 113 102 112 63 190 147 110 281 230 260 323 238 168 156 168 192 179 65 133 127 133 39 0 81 216 259 307 253
    72 67 100 90 92 78 76 36 159 184 150 153 140 90 32 22 33 33 115 66 113 231 194 197 278 166 92 99 115 142 136 54 98 137 127 100 81 0 193 214 253 187
    126 139 112 122 129 169 148 160 110 99 137 176 168 205 200 202 208 173 160 200 108 100 36 126 120 156 162 111 94 79 67 158 95 100 85 190 216 193 0 74 129 137
    141 148 114 126 127 163 147 178 72 46 100 137 134 193 209 215 218 205 138 197 164 26 39 64 65 111 150 116 98 72 81 195 127 163 143 241 259 214 74 0 55 80
    183 186 153 163 161 191 180 218 95 69 115 143 145 214 243 250 251 249 159 224 217 30 94 64 49 115 176 155 140 115 129 243 175 218 197 292 307 253 129 55 0 81
    124 122 94 101 95 115 111 153 35 38 37 62 64 135 171 179 178 191 80 147 184 75 103 19 124 34 101 97 90 74 95 190 132 194 170 246 253 187 137 80 81 0
"""


problem = cargar_problema("swiss42")
Nodos = list(problem.get_nodes())

assert len(Nodos) == 42 and problem.get_weight(0, 1) == 15
print(f"\nInstancia '{problem.nombre}': {len(Nodos)} ciudades")

Datos leídos del fichero local 'swiss42.tsp'

Instancia 'swiss42': 42 ciudades


---
## Funciones de la Actividad Guiada 3

Las mismas que en el notebook principal: una solución es la lista de ciudades en orden de visita,
empezando siempre por la 0.

In [2]:
def crear_solucion(Nodos):
    """Genera una solución aleatoria que empieza en el nodo 0."""
    resto = Nodos[1:]
    random.shuffle(resto)
    return [Nodos[0]] + resto


def distancia(a, b, problem):
    """Distancia entre dos ciudades."""
    return problem.get_weight(a, b)


def distancia_total(solucion, problem):
    """Longitud total del recorrido, incluido el regreso a la ciudad de partida."""
    total = 0
    for i in range(len(solucion) - 1):
        total += distancia(solucion[i], solucion[i + 1], problem)
    return total + distancia(solucion[len(solucion) - 1], solucion[0], problem)


def es_solucion_valida(solucion, problem):
    """Comprueba que el recorrido visita todas las ciudades una vez y empieza en la 0."""
    nodos = list(problem.get_nodes())
    return (len(solucion) == len(nodos)
            and solucion[0] == nodos[0]
            and sorted(solucion) == sorted(nodos))


print("Solución de ejemplo:", distancia_total(crear_solucion(Nodos), problem))

Solución de ejemplo: 4543


---
## Funciones auxiliares

Un algoritmo genético mantiene una **población** de soluciones y la hace evolucionar por
generaciones: los individuos se **cruzan** para producir descendencia, ésta **muta** con cierta
probabilidad, y una **selección** devuelve la población a su tamaño quedándose con los más aptos.
Aquí la aptitud es la longitud del recorrido, y cuanto menor, mejor.

El punto delicado en el TSP es que **un cruce no produce recorridos válidos**. Si se corta por la
mitad y se pegan los trozos de dos padres, algunas ciudades aparecen dos veces y otras desaparecen.
De ahí `Factibilizar`, que repara al hijo conservando el orden heredado y añadiendo al final las
ciudades que faltan.

In [3]:
def generar_poblacion(Nodos, N):
    """Genera una población inicial de N soluciones aleatorias."""
    return [crear_solucion(Nodos) for _ in range(N)]


def Evaluar_Poblacion(poblacion, problem):
    """Devuelve (mejor_individuo, su_distancia) de la población."""
    mejor = min(poblacion, key=lambda individuo: distancia_total(individuo, problem))
    return mejor, distancia_total(mejor, problem)


def Factibilizar(solucion, problem):
    """Repara un hijo del cruce para que vuelva a ser un recorrido válido.

    Tras un cruce de un punto hay ciudades repetidas y ciudades ausentes. Se recorre el hijo
    conservando la primera aparición de cada ciudad —así se hereda el orden de los padres— y
    se añaden al final, en orden aleatorio, las que falten.
    """
    nodos = list(problem.get_nodes())

    vistas = set()
    reparada = []
    for ciudad in solucion:
        if ciudad not in vistas:
            reparada.append(ciudad)
            vistas.add(ciudad)

    faltan = [ciudad for ciudad in nodos if ciudad not in vistas]
    random.shuffle(faltan)
    reparada.extend(faltan)

    # El recorrido debe empezar en la ciudad de partida
    if reparada[0] != nodos[0]:
        posicion = reparada.index(nodos[0])
        reparada[0], reparada[posicion] = reparada[posicion], reparada[0]

    return reparada


def Mutar(solucion, mutacion, operador="inversion"):
    """Muta un individuo con probabilidad `mutacion`.

    Dos operadores, según sugiere el comentario del enunciado:
      - "intercambio": permuta dos ciudades (el que propone el esqueleto)
      - "inversion"  : da la vuelta al segmento entre dos posiciones
    """
    if random.random() >= mutacion:
        return solucion

    i, j = sorted(random.sample(range(1, len(solucion)), 2))

    if operador == "inversion":
        return solucion[:i] + solucion[i:j + 1][::-1] + solucion[j + 1:]

    mutada = solucion[:]
    mutada[i], mutada[j] = mutada[j], mutada[i]
    return mutada


def Descendencia(padres, problem, mutacion, puntos=1, operador="inversion"):
    """Genera dos hijos a partir de dos padres, por cruce de uno o dos puntos."""
    padre1, padre2 = padres
    longitud = len(padre1)

    if puntos == 2:
        i, j = sorted(random.sample(range(1, longitud), 2))
        hijo1 = padre1[:i] + padre2[i:j] + padre1[j:]
        hijo2 = padre2[:i] + padre1[i:j] + padre2[j:]
    else:
        corte = random.randint(1, longitud - 1)
        hijo1 = padre1[:corte] + padre2[corte:]
        hijo2 = padre2[:corte] + padre1[corte:]

    # Los hijos del cruce no son recorridos válidos: hay que repararlos antes de mutarlos
    return [Mutar(Factibilizar(hijo, problem), mutacion, operador) for hijo in (hijo1, hijo2)]


def torneo(poblacion, problem, tamano=3):
    """Selección previa al cruce: se sortean `tamano` individuos y gana el mejor.

    Es el «proceso previo de selección» que sugiere el comentario de Cruzar. Da ventaja a los
    buenos sin excluir del todo a los demás, que es lo que mantiene la diversidad.
    """
    aspirantes = random.sample(poblacion, min(tamano, len(poblacion)))
    return min(aspirantes, key=lambda individuo: distancia_total(individuo, problem))


def Cruzar(poblacion, mutacion, problem, puntos=1, operador="inversion", usar_torneo=True):
    """Cruza la población y la devuelve ampliada con los hijos."""
    ampliada = list(poblacion)

    for _ in range(len(poblacion) // 2):
        if usar_torneo:
            padres = (torneo(poblacion, problem), torneo(poblacion, problem))
        else:
            padres = tuple(random.sample(poblacion, 2))
        ampliada.extend(Descendencia(padres, problem, mutacion, puntos, operador))

    return ampliada


def Seleccionar(problem, poblacion, N, elitismo):
    """Reduce la población a N individuos.

    Se conserva intacta la élite —la fracción `elitismo` de los mejores—, y el resto de plazas
    se cubren por **ruleta**: cada individuo tiene probabilidad proporcional a su aptitud, que
    aquí es 1/distancia. Así los mediocres conservan una oportunidad y no se pierde diversidad.
    """
    evaluada = sorted(((distancia_total(individuo, problem), individuo) for individuo in poblacion),
                      key=lambda par: par[0])

    n_elite = max(1, int(N * elitismo))
    elegidos = [individuo for _, individuo in evaluada[:n_elite]]
    resto = evaluada[n_elite:]

    plazas = N - len(elegidos)
    if plazas > 0 and resto:
        pesos = [1.0 / d for d, _ in resto]
        total = sum(pesos)

        for _ in range(min(plazas, len(resto))):
            umbral = random.random() * total
            acumulado = 0.0
            for k, (_, individuo) in enumerate(resto):
                acumulado += pesos[k]
                if acumulado >= umbral:
                    elegidos.append(individuo)
                    total -= pesos[k]
                    pesos[k] = 0.0        # Ya seleccionado: no puede repetirse
                    break

    return elegidos[:N]


# --- Verificación de las piezas -----------------------------------------------------------------
random.seed(1)
poblacion_prueba = generar_poblacion(Nodos, 10)
assert len(poblacion_prueba) == 10
assert all(es_solucion_valida(individuo, problem) for individuo in poblacion_prueba)

mejor, distancia_mejor = Evaluar_Poblacion(poblacion_prueba, problem)
assert distancia_mejor == min(distancia_total(i, problem) for i in poblacion_prueba)

# Un hijo del cruce, antes de reparar, no es un recorrido válido; después sí
padre1, padre2 = poblacion_prueba[0], poblacion_prueba[1]
hijo_roto = padre1[:20] + padre2[20:]
assert not es_solucion_valida(hijo_roto, problem), "el cruce debería romper la solución"
assert es_solucion_valida(Factibilizar(hijo_roto, problem), problem)

# Los dos operadores de mutación conservan la validez
for operador in ("intercambio", "inversion"):
    assert es_solucion_valida(Mutar(padre1, 1.0, operador), problem)

assert all(es_solucion_valida(h, problem)
           for h in Descendencia((padre1, padre2), problem, 0.5))

# Cruzar amplía la población, y Seleccionar la devuelve al tamaño N
ampliada = Cruzar(poblacion_prueba, 0.3, problem)
assert len(ampliada) == 20
reducida = Seleccionar(problem, ampliada, 10, 0.2)
assert len(reducida) == 10
assert all(es_solucion_valida(individuo, problem) for individuo in reducida)

print("Verificado: las siete funciones cumplen su contrato y toda la población es válida.")

Verificado: las siete funciones cumplen su contrato y toda la población es válida.


---
## Proceso principal

El ciclo de generaciones: cruzar, seleccionar, evaluar y repetir. La única diferencia respecto al
esqueleto es que se guarda la mejor solución **de todas las generaciones**, y no solo la de la
última población: el elitismo la protege casi siempre, pero así queda garantizado.

In [4]:
def algoritmo_genetico(problem=problem, N=100, mutacion=.15, elitismo=.1, generaciones=100,
                       puntos=1, operador="inversion", usar_torneo=True, traza=25):
    """Algoritmo genético para el TSP.

    problem      = datos del problema
    N            = tamaño de la población
    mutacion     = probabilidad de mutar un individuo
    elitismo     = porción de los mejores que pasa intacta a la siguiente generación
    generaciones = criterio de parada
    puntos       = puntos de corte del cruce (1 o 2)
    operador     = operador de mutación ("intercambio" o "inversion")
    usar_torneo  = seleccionar por torneo a quién se cruza
    traza        = cada cuántas generaciones se informa (0 para no informar)
    """
    Nodos = list(problem.get_nodes())
    poblacion = generar_poblacion(Nodos, N)

    mejor_solucion, mejor_distancia = Evaluar_Poblacion(poblacion, problem)

    n = 0
    parar = False
    while not parar:
        poblacion = Cruzar(poblacion, mutacion, problem, puntos, operador, usar_torneo)
        poblacion = Seleccionar(problem, poblacion, N, elitismo)

        solucion, distancia_actual = Evaluar_Poblacion(poblacion, problem)
        if distancia_actual < mejor_distancia:
            mejor_solucion, mejor_distancia = solucion, distancia_actual

        if traza and n % traza == 0:
            print(f"Generación {n:>4} | mejor de la población: {distancia_actual:>6} "
                  f"| mejor global: {mejor_distancia:>6}")

        if n == generaciones:
            parar = True
        n += 1

    return mejor_solucion, mejor_distancia


random.seed(42)
inicio = time.perf_counter()
solucion, distancia_final = algoritmo_genetico(problem=problem, N=500, mutacion=.3,
                                               elitismo=.40, generaciones=250, traza=50)

assert es_solucion_valida(solucion, problem)
print(f"\nMejor solución: {solucion}")
print(f"Distancia     : {distancia_final}   (en {time.perf_counter() - inicio:.1f} s)")

Generación    0 | mejor de la población:   3723 | mejor global:   3723
Generación   50 | mejor de la población:   1569 | mejor global:   1569
Generación  100 | mejor de la población:   1378 | mejor global:   1378
Generación  150 | mejor de la población:   1355 | mejor global:   1355
Generación  200 | mejor de la población:   1355 | mejor global:   1355
Generación  250 | mejor de la población:   1355 | mejor global:   1355

Mejor solución: [0, 32, 34, 33, 20, 31, 35, 36, 17, 37, 15, 16, 14, 19, 13, 5, 26, 18, 12, 11, 25, 10, 41, 23, 21, 40, 24, 39, 22, 38, 9, 8, 29, 30, 28, 27, 2, 3, 4, 6, 1, 7]
Distancia     : 1355   (en 14.6 s)


---
## Qué aporta cada mejora

Los comentarios del esqueleto sugieren cuatro cosas que «se podrían» hacer. Conviene comprobar
cuáles importan de verdad en lugar de darlas por buenas. **Tres de las cuatro** se pueden aislar
con un interruptor el operador de mutación, los puntos de corte y el torneo previo, y se miden
abajo ejecutando cada variante con la **misma semilla y el mismo presupuesto**, cambiando una
sola cosa cada vez.

La cuarta, la ruleta para los individuos no élite, está implementada dentro de `Seleccionar`
pero **no se mide**: aislarla exigiría reescribir esa función. Queda razonada y sin comprobar,
y así se dice, en vez de atribuirle un efecto que no se ha observado.

Se usa una configuración más pequeña que la de arriba (N=200, 120 generaciones) para que la
comparación no tarde de más; lo que interesa es la diferencia relativa, no el valor absoluto.

In [6]:
OPTIMO_SWISS42 = 1273     # Óptimo conocido de la instancia, publicado en TSPLIB

CONFIGURACION = {"N": 200, "mutacion": .3, "elitismo": .2, "generaciones": 120, "traza": 0}

variantes = [
    ("Base: inversión, 1 punto, torneo", {}),
    ("Mutación por intercambio",           {"operador": "intercambio"}),
    ("Cruce de 2 puntos",                  {"puntos": 2}),
    ("Sin torneo (parejas al azar)",       {"usar_torneo": False}),
]

print(f"{'variante':<38} | {'distancia':>10} | {'sobre el óptimo':>16} | {'seg':>6}")
print("-" * 80)

medidas = {}
for etiqueta, cambios in variantes:
    random.seed(2024)                      # Misma semilla: se compara lo mismo
    argumentos = dict(CONFIGURACION)
    argumentos.update(cambios)

    inicio = time.perf_counter()
    _, d = algoritmo_genetico(problem=problem, **argumentos)
    segundos = time.perf_counter() - inicio

    medidas[etiqueta] = d
    exceso = (d / OPTIMO_SWISS42 - 1) * 100
    print(f"{etiqueta:<38} | {d:>10} | {exceso:>15.1f}% | {segundos:>6.1f}")

base = medidas["Base: inversión, 1 punto, torneo"]
con_intercambio = medidas["Mutación por intercambio"]

print(f"\nLa mutación por inversión mejora a la de intercambio en "
      f"{con_intercambio - base} unidades ({(1 - base / con_intercambio) * 100:.0f} %).")

# Las dos mejoras que de verdad deciden el resultado
assert base < con_intercambio, "la mutación por inversión debería ganar a la de intercambio"
assert base < medidas["Sin torneo (parejas al azar)"], "el torneo previo debería aportar"

variante                               |  distancia |  sobre el óptimo |    seg
--------------------------------------------------------------------------------
Base: inversión, 1 punto, torneo       |       1397 |             9.7% |    2.6
Mutación por intercambio               |       1787 |            40.4% |    2.4
Cruce de 2 puntos                      |       1507 |            18.4% |    2.7
Sin torneo (parejas al azar)           |       2110 |            65.8% |    1.6

La mutación por inversión mejora a la de intercambio en 390 unidades (22 %).


### Por qué la mutación por inversión gana tan claramente

No es casualidad, y tiene la misma explicación que en el apartado de búsqueda local de la AG3.

Intercambiar dos ciudades de posición toca **cuatro aristas** del recorrido: se rompen las dos
conexiones de cada una de las dos ciudades. Es un cambio brusco, que casi siempre estropea un
recorrido ya decente.

Invertir el segmento entre dos posiciones toca solo **dos aristas**, porque el interior del tramo
se recorre al revés pero sigue conectado igual. Es exactamente el movimiento que **deshace los
cruces** del recorrido, que son la causa habitual de que un ciclo sea largo.

En un algoritmo genético eso importa el doble: la mutación no solo explora, también tiene que
**conservar lo bueno que la población ya ha encontrado**. Un operador que destruye recorridos
buenos convierte la mutación en ruido, y la población deja de mejorar.

---
## Conclusiones

| Mejora sugerida por el enunciado | Efecto |
|---|---|
| **Torneo previo al cruce** | **Decisivo.** Es la variante que más se degrada al quitarla: emparejar al azar multiplica por varias veces la distancia al óptimo |
| **Operador de mutación** (inversión frente a intercambio) | **Decisivo.** El intercambio cuadruplica el exceso sobre el óptimo |
| Cruce de dos puntos frente a uno | **No compensa** en este problema: empeora respecto al de un punto |
| Ruleta para los no élite | **No medida.** Está implementada, pero aislarla exigiría reescribir `Seleccionar`. Su papel es mantener la diversidad y evitar la convergencia prematura |

De las tres que se han podido aislar, **dos resultan decisivas** y una **empeora** el resultado;
la cuarta queda implementada pero sin comprobar. Vale la pena señalarlo: los comentarios del
esqueleto dicen «se podría», no «hay que», y medirlas era la única forma de saber cuáles
merecían la pena.

**Lo que se aprende:**

1. **En el TSP, el cruce clásico no encaja bien.** Cortar y pegar dos recorridos no produce un
   recorrido, y hay que repararlo. Esa reparación destruye parte de la información heredada, que es
   justamente lo que un cruce debería preservar. Por eso aquí acaba pesando más la mutación que el
   cruce, al revés de lo habitual en un algoritmo genético.

2. **El elitismo garantiza que no se retrocede**, pero en exceso reduce la diversidad y la población
   converge antes de tiempo. La ruleta para las plazas no reservadas a la élite es lo que compensa
   ese efecto.

3. **La misma lección que en el resto de la AG3.** Sea búsqueda local, recocido simulado o un
   algoritmo genético, lo que más determina el resultado es **cómo se define el movimiento** entre
   soluciones. Cambiar el operador ha valido más que cambiar de metaheurística.

4. **Ninguna de estas técnicas demuestra nada.** Devuelven la mejor solución que han sabido
   encontrar, sin garantía de que sea la óptima. Tener una referencia —aquí el óptimo conocido
   1273— es lo único que permite saber si un resultado es bueno o solo es el mejor que hemos visto.

---
## Uso de inteligencia artificial

La inteligencia artificial fue utilizada como herramienta de apoyo en la redacción del presente trabajo, principalmente para mejorar la claridad de las ideas, corregir aspectos de redacción y evitar la repetición o redundancia de palabras y expresiones. El contenido, análisis y conclusiones presentados son responsabilidad del autor.